# Project 3 — Local Meeting Notes Summarizer
## Summarize Transcripts into Actions, Decisions, and Blockers

**What you'll learn:**
- Structured summarization with LangChain
- Output parsing into defined sections
- Chain-of-thought extraction of action items
- Iterative refinement with map-reduce

**Stack:** Ollama · LangChain · Pydantic · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pydantic

## Step 1 — Set Up Local LLM

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:8b", temperature=0.1)
print("LLM ready for summarization!")

## Step 2 — Create Sample Meeting Transcripts

In [ ]:
transcripts = {
    "sprint_planning": """
Sprint Planning Meeting — 2025-01-15
Attendees: Alice (PM), Bob (Backend), Carol (Frontend), Dave (QA)

Alice: Let's review the backlog. We have 15 tickets for this sprint.
Bob: The API refactoring is almost done. I need 2 more days for the auth module.
Carol: The dashboard redesign is blocked by the new API endpoints. Bob, can you
prioritize the /users endpoint?
Bob: Sure, I'll have that ready by Wednesday.
Dave: I found 3 critical bugs in the payment flow. They need to be fixed before release.
Alice: Let's mark those as P0. Bob, can you look at the payment bugs first?
Bob: I can start on that tomorrow.
Carol: I also need the design tokens from the design team. I'll follow up today.
Alice: Good. Let's aim to have all P0 items done by Thursday. Sprint demo is Friday.
Dave: I'll prepare the test cases for the new endpoints by Wednesday.
""",
    "stakeholder_review": """
Stakeholder Review — 2025-01-20
Attendees: VP Product, Engineering Lead, Marketing, Support Lead

VP Product: Q4 revenue grew 18%. The new pricing tier drove most of the growth.
Engineering: We shipped the real-time analytics feature. Latency is under 200ms.
Marketing: The launch campaign reached 50K users. Conversion rate was 3.2%.
Support Lead: Ticket volume increased 25% after the launch. Most issues are about
the new billing system. We need better documentation.
VP Product: Let's prioritize help docs this sprint. Can engineering add tooltips?
Engineering: Yes, we can add contextual help. Need 1 week.
Marketing: We should also update the knowledge base articles.
VP Product: Decision: Prioritize documentation and tooltips over new features next sprint.
"""
}
print(f"Created {len(transcripts)} sample transcripts")

## Step 3 — Structured Summarization Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

summary_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a meeting summarizer. Extract a structured summary with:

## Summary
(2-3 sentence overview)

## Key Decisions
- (each decision made)

## Action Items
- [ ] (task) — Owner: (person) — Due: (date if mentioned)

## Blockers
- (any blockers or dependencies identified)

## Follow-ups Needed
- (items requiring follow-up)

Be concise and factual. Only include information explicitly stated."""),
    ("human", "Summarize this meeting transcript:\n\n{transcript}")
])

summary_chain = summary_prompt | llm | StrOutputParser()

for name, transcript in transcripts.items():
    print(f"\n{'='*70}")
    print(f"Meeting: {name}")
    print('='*70)
    result = summary_chain.invoke({"transcript": transcript})
    print(result)

## Step 4 — Pydantic-Structured Output

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class ActionItem(BaseModel):
    task: str = Field(description="The specific task to be done")
    owner: str = Field(description="Person responsible")
    due_date: Optional[str] = Field(None, description="Due date if mentioned")
    priority: str = Field(default="normal", description="P0/P1/normal")

class MeetingSummary(BaseModel):
    title: str = Field(description="Meeting title")
    summary: str = Field(description="2-3 sentence overview")
    decisions: list[str] = Field(description="Key decisions made")
    action_items: list[ActionItem] = Field(description="Action items with owners")
    blockers: list[str] = Field(description="Blockers identified")

structured_llm = llm.with_structured_output(MeetingSummary)

result = structured_llm.invoke(
    f"Extract a structured summary from this meeting:\n\n{transcripts['sprint_planning']}"
)
print(f"Title: {result.title}")
print(f"Summary: {result.summary}")
print(f"\nDecisions:")
for d in result.decisions:
    print(f"  • {d}")
print(f"\nAction Items:")
for ai in result.action_items:
    print(f"  [{ai.priority}] {ai.task} — Owner: {ai.owner} — Due: {ai.due_date}")
print(f"\nBlockers:")
for b in result.blockers:
    print(f"  ⚠ {b}")

## Step 5 — Batch Processing with Map-Reduce

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Map: summarize each transcript individually
map_prompt = ChatPromptTemplate.from_messages([
    ("system", "Summarize this meeting into 3-5 bullet points of key outcomes."),
    ("human", "{transcript}")
])
map_chain = map_prompt | llm | StrOutputParser()

# Reduce: combine individual summaries
reduce_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are given summaries from multiple meetings. Produce a
consolidated weekly report with: Overall Progress, Combined Action Items,
Cross-cutting Blockers, and Priorities for Next Week."""),
    ("human", "Individual meeting summaries:\n\n{summaries}")
])
reduce_chain = reduce_prompt | llm | StrOutputParser()

# Execute map-reduce
individual_summaries = []
for name, transcript in transcripts.items():
    summary = map_chain.invoke({"transcript": transcript})
    individual_summaries.append(f"### {name}\n{summary}")
    print(f"Summarized: {name}")

combined = "\n\n".join(individual_summaries)
weekly_report = reduce_chain.invoke({"summaries": combined})
print(f"\n{'='*70}")
print("WEEKLY CONSOLIDATED REPORT")
print('='*70)
print(weekly_report)

## What You Learned
- **Structured prompts** for meeting summarization
- **Pydantic output parsing** for machine-readable summaries
- **Map-reduce pattern** for multi-document summarization
- **Action item extraction** with metadata (owner, priority, due date)